<a href="https://colab.research.google.com/github/suchetindrakanty/Emotion_detection/blob/main/major_project_dataset_trail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>    
<h3>American Association of Physicists in Medicine</h3>    
<h3>Grand Challenge 2020</h3>
<h3>OpenKBP</h3>
<hr>
<h1>Introduction for Google Colab</h1>
<h3>January 25, 2022</h3>
</center>

# Setup

Before running this notebook, we need to get the repo which contains the data. The download should be quick as it's a sercer-to-server process.

In [18]:
# Get the repo
repo_dir = 'open-kbp'
!git clone https://github.com/ababier/open-kbp.git {repo_dir}

fatal: destination path 'open-kbp' already exists and is not an empty directory.


In [19]:
# Add repo to path
import sys
sys.path.append(repo_dir)

Also let's install the required libraries.

In [20]:
!sed -i 's/scikit_image==0.19.3/scikit-image==0.22.0/' open-kbp/requirements_filtered.txt
!pip install -r open-kbp/requirements_filtered.txt --upgrade --no-cache-dir



# Create a filtered requirements file to avoid conflicts with pre-installed tensorflow/keras
with open('open-kbp/requirements_filtered.txt', 'w') as f:
    f.write('black==22.12.0\n')
    f.write('h5py==3.8.0\n')
    f.write('isort==5.11.4\n')
    f.write('jupyter==1.0.0\n')
    f.write('matplotlib==3.6.3\n')
    f.write('more_itertools==9.0.0\n')
    f.write('pandas==1.5.3\n')
    f.write('pylint==2.16.2\n')
    f.write('scikit_image==0.22.0\n')
    f.write('scipy==1.10.0\n')

!pip install -r open-kbp/requirements_filtered.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 84.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 298.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python >=3.8,<3.12; 1.10.0rc1 Requires-Python >=3.8,<3.12; 1.10.0rc2 Requires-Python >=3.8,<3.12; 1.10.1 Requires-Python >=3.8,<3.12; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10;

Import all necessary packages for the notebook.

In [23]:
# === Patch all keras / TensorFlow import issues in open-kbp ===

# 1️⃣ Fix OptimizerV2 → Optimizer and KerasTensor → tf.Tensor
!sed -i 's/OptimizerV2/Optimizer/' open-kbp/provided_code/network_architectures.py
!sed -i 's/KerasTensor/tf.Tensor/g' open-kbp/provided_code/network_architectures.py

# 2️⃣ Add missing TensorFlow import (for tf.Tensor)
!sed -i '1i import tensorflow as tf' open-kbp/provided_code/network_architectures.py

# 3️⃣ Fix Adam optimizer import in network_functions.py
!sed -i 's/from keras.optimizers.optimizer_v2.adam import Adam/from tensorflow.keras.optimizers import Adam/' open-kbp/provided_code/network_functions.py

# 4️⃣ Remove broken KerasTensor import & update BatchNormalization path
!sed -i '/from keras.engine.keras_tensor import KerasTensor/d' open-kbp/provided_code/network_architectures.py
!sed -i 's/from keras.layers.normalization.batch_normalization import BatchNormalization/from tensorflow.keras.layers import BatchNormalization/' open-kbp/provided_code/network_architectures.py

# 5️⃣ (Optional) Check first 20 lines to confirm TensorFlow import
!head -n 20 open-kbp/provided_code/network_architectures.py

# 6️⃣ Verify the architecture file contents
!echo -e "\n✅ Network architectures.py after patching:\n"
!grep -E "import|Optimizer|Tensor" open-kbp/provided_code/network_architectures.py

# 7️⃣ Load TensorFlow and try importing your project modules
import shutil
from pathlib import Path
import tensorflow as tf

from provided_code.data_loader import DataLoader
from provided_code.dose_evaluation_class import DoseEvaluator
from provided_code.network_functions import PredictionModel
from provided_code.utils import get_paths

print("\n✅ All imports successful — patch completed successfully!")


import tensorflow as tf
""" Neural net architectures """
from typing import Optional

from keras.layers import Activation, AveragePooling3D, Conv3D, Conv3DTranspose, Input, LeakyReLU, SpatialDropout3D, concatenate
from tensorflow.keras.layers import BatchNormalization
from keras.models import Model
from tensorflow.keras.optimizers import Optimizer

from provided_code.data_shapes import DataShapes


class DefineDoseFromCT:
    """This class defines the architecture for a U-NET and must be inherited by a child class that
    executes various functions like training or predicting"""

    def __init__(
        self,
        data_shapes: DataShapes,
        initial_number_of_filters: int,

✅ Network architectures.py after patching:

import tensorflow as tf
from typing import Optional
from keras.layers import Activation, AveragePooling3D, Conv3D, Conv3DTranspose, Input, LeakyReLU, SpatialDropout3D, concatenate
from tensorflow.keras.layers import BatchNormalization
from keras.models import Mo

The functions loaded from _provided\_code_ are written for this competition, and you can access them via the file
explorer on the left hand side of the Colab window. You're welcome to change them as much as
you'd like. If you use Google Drive now, keep in mind, however, that on Colab any changes you make to the files in your Google Drive will only be recognized by Colab when the _Runtime_ is restarted via the Restart

 Runtime option in the top toolbar. If you implement a neural network, we urge you to you start with the provided
 network architecture and network functions. The neural network we provide is only meant to be a template, and will not
 be a competitive model without some significant modifications.



# Data loading
Before we run anything, first define the paths where the provided data is stored and where the results (e.g., models, predictions) should be saved.

In [24]:
# Define project directories
primary_directory = Path(repo_dir).resolve()  # directory where everything is stored
provided_data_dir = primary_directory / "provided-data"
training_data_dir = provided_data_dir / "train-pats"
validation_data_dir = provided_data_dir / "validation-pats"
testing_data_dir = provided_data_dir / "test-pats"
results_dir = primary_directory / "results"  # where any data generated by this code (e.g., predictions, models) are stored

Name the model. This name will be used to label directories containing the results that the model generates. Also,
define how many epochs the model should be trained for. It will likely take a large number of epochs (e.g., 100-200)
to get good results.

In [25]:
test_time = False  # Only change this to True when the model has been fully tuned on the validation set
prediction_name = "baseline"  # Name model to train and number of epochs to train it for
num_epochs = 2

Retrieve the paths for all patient directories in the training set and seperate them into a list of paths for training
a model and another for hold-out testing.

In [26]:
# Prepare the data directory
training_plan_paths = get_paths(training_data_dir)  # gets the path of each plan's directory

# Model training

Initialize a data loader for the training set data, and use it to initialize a prediction model object. Call the
train_model method to train the model for the predefined number of epochs.

In [ ]:
# Train a model
data_loader_train = DataLoader(training_plan_paths)
dose_prediction_model_train = PredictionModel(data_loader_train, results_dir, prediction_name,  "train")
dose_prediction_model_train.train_model(epochs=num_epochs, save_frequency=1, keep_model_history=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 10)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 128,  │          0 │ input_layer[0][0… │
│ (Concatenate)       │ 128, 11)          │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d (Conv3D)     │ (None, 64, 64,    │        704 │ concatenate[0][0] │
│                     │ 64, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │          4 │ conv3d[0][0]      │
│ (BatchNormalizatio… │ 64, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 64, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_1 (Conv3D)   │ (None, 32, 32,    │        128 │ leaky_re_lu[0][0] │
│                     │ 32, 2)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │          8 │ conv3d_1[0][0]    │
│ (BatchNormalizatio… │ 32, 2)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 32, 2)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_2 (Conv3D)   │ (None, 16, 16,    │        512 │ leaky_re_lu_1[0]… │
│                     │ 16, 4)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │         16 │ conv3d_2[0][0]    │
│ (BatchNormalizatio… │ 16, 4)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 16, 4)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_3 (Conv3D)   │ (None, 8, 8, 8,   │      2,048 │ leaky_re_lu_2[0]… │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 8,   │         32 │ conv3d_3[0][0]    │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_3       │ (None, 8, 8, 8,   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_4 (Conv3D)   │ (None, 4, 4, 4,   │      4,096 │ leaky_re_lu_3[0]… │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 4, 4,   │         32 │ conv3d_4[0][0]  

 Total params: 29,561 (115.47 KB)

 Trainable params: 29,469 (115.11 KB)

 Non-trainable params: 92 (368.00 B)

Beginning epoch 0


0it [00:00, ?it/s]

Note that during training we will only keep models that are __save_frequency * keep_model_history__ epochs back from the
current epoch. We do this because models are very large (~1 GB).

Now that the model is trained we can use it to predict the dose for a set of hold-out patients from the validation or
testing set. The code block below gets the paths of all plans in the hold out set you selected earlier.


In [ ]:
# Define hold out set
hold_out_data_dir = validation_data_dir if test_time is False else testing_data_dir
stage_name, _ = hold_out_data_dir.stem.split("-")
hold_out_plan_paths = get_paths(hold_out_data_dir)

# Model testing

We start by making a new data loader for the held-out set, and use it to predict (and save) a
set of out-of-sample dose distributions. Note that we change the mode of the data loader to 'dose_prediction' to
load only the data needed to make a prediction.


In [ ]:
# Predict dose for the held out set
data_loader_hold_out = DataLoader(hold_out_plan_paths)
dose_prediction_model_hold_out = PredictionModel(data_loader_hold_out, results_dir, model_name=prediction_name, stage=stage_name)
dose_prediction_model_hold_out.predict_dose(epoch=num_epochs)

Load each predicted dose distribution and evaluate it against the ground truth using the
competition metrics.

In [ ]:
 # Evaluate dose metrics
data_loader_hold_out_eval = DataLoader(hold_out_plan_paths)
prediction_paths = get_paths(dose_prediction_model_hold_out.prediction_dir, extension="csv")
hold_out_prediction_loader = DataLoader(prediction_paths)
dose_evaluator = DoseEvaluator(data_loader_hold_out_eval, hold_out_prediction_loader)

# print out scores if data was left for a hold out set
if not data_loader_hold_out_eval.patient_paths:
    print("No patient information was given to calculate metrics")
else:
    dose_evaluator.evaluate()
    dvh_score, dose_score = dose_evaluator.get_scores()
    print(f"For this out-of-sample test on {stage_name}:\n\tthe DVH score is {dvh_score:.3f}\n\tthe dose score is {dose_score:.3f}")

# Saving results

Once you're happy with your dose distributions you can zip up the predictions with the code block below. The zipped file
will contain the dose distributions for the validation set. It can be uploaded directly to CodaLab.

In [ ]:
# Zip dose to submit
submission_dir = results_dir / "submissions"
submission_dir.mkdir(exist_ok=True)
submission_zipfile = shutil.make_archive(
    str(submission_dir / prediction_name),
    "zip",
    dose_prediction_model_hold_out.prediction_dir
)

You can save the model and submission to Google Drive or download them (on the left hand panel, click `...` -> `Download`).

In [ ]:
# Mount your personal google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Save the submission
submissions_on_drive = Path('/content/drive/MyDrive/open-kbp-subissions')
submissions_on_drive.mkdir(exist_ok=True)
shutil.copy(submission_zipfile, submissions_on_drive / f'{prediction_name}.zip')

> Note that the model will be lost once you close the Colab session. You can download or save what's in the `results` folder.